# SPINE-GPE v7 — Dossiê de Reprodutibilidade da Fase 0 v1.0.0

Este notebook valida a cadeia certificada da Fase 0 e cria um dossiê portátil, com manifestos, ambiente, scripts, relatórios, hashes, ZIP, lock e freeze. Nenhuma estimativa científica é recalculada.

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Mounted at /content/drive


In [3]:
from pathlib import Path
import shutil, zipfile, os

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
PACKAGE_ZIP = Path("/content/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip")
INSTALL_DIR = ROOT / "scripts" / "phase0_reproducibility_dossier_v100"
EXPECTED_MASTER_LOCK_SHA256 = "38d3e5e5380032fd61997a8e913113d5430f8670af9c51870a47edcc97232c53"
EXPECTED_MASTER_FREEZE_SHA256 = "3ee7e7eca7f71d6e4216621e30bb71541829fa5709be13d2a4ffb46155312454"
RAW_HASH_MODE = "all"  # arquivístico; use "declared" apenas para um teste rápido
MAX_EMBED_MB = 256

assert PACKAGE_ZIP.is_file(), f"Envie o pacote para {PACKAGE_ZIP}"
if INSTALL_DIR.exists():
    shutil.rmtree(INSTALL_DIR)
INSTALL_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as zf:
    zf.extractall(INSTALL_DIR)
print("Instalado em:", INSTALL_DIR)
print("Arquivos:", len(list(INSTALL_DIR.rglob("*"))))


AssertionError: Envie o pacote para /content/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip

In [8]:
import subprocess, sys

ENGINE = INSTALL_DIR / "SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_v1.0.0.py"
assert ENGINE.is_file(), ENGINE

cmd_audit = [
    sys.executable, str(ENGINE),
    "--root", str(ROOT),
    "--mode", "audit",
    "--run-id", "phase0_reproducibility_audit_v100",
    "--expected-master-lock-sha256", EXPECTED_MASTER_LOCK_SHA256,
    "--expected-master-freeze-sha256", EXPECTED_MASTER_FREEZE_SHA256,
    "--raw-hash-mode", "declared",
    "--strict",
]
print("Executando audit:")
print(" ".join(cmd_audit))
print()
proc = subprocess.Popen(cmd_audit, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
audit_exit = proc.wait()
print("\nAudit exit code:", audit_exit)
assert audit_exit == 0


Executando audit:
/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_reproducibility_dossier_v100/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_v1.0.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode audit --run-id phase0_reproducibility_audit_v100 --expected-master-lock-sha256 38d3e5e5380032fd61997a8e913113d5430f8670af9c51870a47edcc97232c53 --expected-master-freeze-sha256 3ee7e7eca7f71d6e4216621e30bb71541829fa5709be13d2a4ffb46155312454 --raw-hash-mode declared --strict

2026-07-26 18:15:26,035 | INFO | SPINE-GPE Phase 0 Reproducibility Dossier v1.0.0 | mode=audit | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-26 18:15:35,589 | INFO | Verificados 25 arquivos declarados
2026-07-26 18:15:42,183 | INFO | Verificados 50 arquivos declarados
2026-07-26 18:15:47,606 | INFO | Verificados 75 arquivos declarados
2026-07-26 18:15:53,193 | INFO | Verificados 100 arquivos declarados
2026-07-26 18:15:59,587 | INFO | Verifica

In [4]:
from pathlib import Path
from google.colab import files
import shutil

PACKAGE_NAME = (
    "SPINE_GPEv7_PHASE0_"
    "REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip"
)

PACKAGE_ZIP = Path("/content") / PACKAGE_NAME

if not PACKAGE_ZIP.is_file():
    print(f"Pacote não encontrado em {PACKAGE_ZIP}")
    print("Selecione agora o ZIP baixado.")

    uploaded = files.upload()

    assert PACKAGE_NAME in uploaded, (
        f"Arquivo esperado: {PACKAGE_NAME}\n"
        f"Arquivos enviados: {list(uploaded)}"
    )

    uploaded_path = Path("/content") / PACKAGE_NAME

    assert uploaded_path.is_file(), uploaded_path

print("Pacote localizado:")
print(PACKAGE_ZIP)
print("Tamanho:", PACKAGE_ZIP.stat().st_size, "bytes")

Pacote não encontrado em /content/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip
Selecione agora o ZIP baixado.


Saving SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip to SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip
Pacote localizado:
/content/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip
Tamanho: 46249 bytes


In [5]:
import hashlib

EXPECTED_SHA256 = (
    "f7e421df26531088f7f570c32883207f"
    "6113a36686b5643a570027bf0c575fd9"
)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

actual_sha256 = sha256_file(PACKAGE_ZIP)

print("SHA-256 esperado:  ", EXPECTED_SHA256)
print("SHA-256 encontrado:", actual_sha256)

assert actual_sha256 == EXPECTED_SHA256, (
    "O ZIP enviado não corresponde ao pacote certificado."
)

print("\nPACKAGE ZIP VERIFIED")

SHA-256 esperado:   f7e421df26531088f7f570c32883207f6113a36686b5643a570027bf0c575fd9
SHA-256 encontrado: f7e421df26531088f7f570c32883207f6113a36686b5643a570027bf0c575fd9

PACKAGE ZIP VERIFIED


In [6]:
from pathlib import Path
import shutil, zipfile, os

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
PACKAGE_ZIP = Path("/content/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip")
INSTALL_DIR = ROOT / "scripts" / "phase0_reproducibility_dossier_v100"
EXPECTED_MASTER_LOCK_SHA256 = "38d3e5e5380032fd61997a8e913113d5430f8670af9c51870a47edcc97232c53"
EXPECTED_MASTER_FREEZE_SHA256 = "3ee7e7eca7f71d6e4216621e30bb71541829fa5709be13d2a4ffb46155312454"
RAW_HASH_MODE = "all"  # arquivístico; use "declared" apenas para um teste rápido
MAX_EMBED_MB = 256

assert PACKAGE_ZIP.is_file(), f"Envie o pacote para {PACKAGE_ZIP}"
if INSTALL_DIR.exists():
    shutil.rmtree(INSTALL_DIR)
INSTALL_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as zf:
    zf.extractall(INSTALL_DIR)
print("Instalado em:", INSTALL_DIR)
print("Arquivos:", len(list(INSTALL_DIR.rglob("*"))))


Instalado em: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_reproducibility_dossier_v100
Arquivos: 16


## Build arquivístico

Com `RAW_HASH_MODE="all"`, esta etapa calcula SHA-256 de todos os arquivos em `01_raw`. Ela pode demorar, mas não copia os dados brutos para o ZIP.

In [9]:
RUN_ID = "phase0_reproducibility_final_v100"
cmd_build = [
    sys.executable, str(ENGINE),
    "--root", str(ROOT),
    "--mode", "build",
    "--run-id", RUN_ID,
    "--expected-master-lock-sha256", EXPECTED_MASTER_LOCK_SHA256,
    "--expected-master-freeze-sha256", EXPECTED_MASTER_FREEZE_SHA256,
    "--raw-hash-mode", RAW_HASH_MODE,
    "--max-embed-mb", str(MAX_EMBED_MB),
    "--strict",
]
print("Executando build:")
print(" ".join(cmd_build))
print()
proc = subprocess.Popen(cmd_build, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
build_exit = proc.wait()
print("\nBuild exit code:", build_exit)
assert build_exit == 0


Executando build:
/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_reproducibility_dossier_v100/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_v1.0.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode build --run-id phase0_reproducibility_final_v100 --expected-master-lock-sha256 38d3e5e5380032fd61997a8e913113d5430f8670af9c51870a47edcc97232c53 --expected-master-freeze-sha256 3ee7e7eca7f71d6e4216621e30bb71541829fa5709be13d2a4ffb46155312454 --raw-hash-mode all --max-embed-mb 256 --strict

2026-07-26 18:18:01,009 | INFO | SPINE-GPE Phase 0 Reproducibility Dossier v1.0.0 | mode=build | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-26 18:18:02,784 | INFO | Verificados 25 arquivos declarados
2026-07-26 18:18:02,946 | INFO | Verificados 50 arquivos declarados
2026-07-26 18:18:03,015 | INFO | Verificados 75 arquivos declarados
2026-07-26 18:18:03,116 | INFO | Verificados 100 arquivos declarados
2026-07-26 18:18:03,198 | I

In [10]:
import hashlib, json

def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

LOCK = ROOT / "00_admin" / "SPINE_GPE_PHASE0_REPRODUCIBILITY_DOSSIER_LOCK.json"
FREEZE = ROOT / "00_admin" / "SPINE_GPE_PHASE0_REPRODUCIBILITY_DOSSIER_FREEZE.json"
assert LOCK.is_file(), LOCK
assert FREEZE.is_file(), FREEZE
lock=json.loads(LOCK.read_text(encoding="utf-8"))
freeze=json.loads(FREEZE.read_text(encoding="utf-8"))
zip_path=Path(lock["dossier_zip"])
assert lock["status"] == "REPRODUCIBILITY_DOSSIER_CERTIFIED"
assert lock["critical_failures"] == []
assert freeze["status"] == "FROZEN"
assert zip_path.is_file()
assert sha256_file(zip_path) == lock["dossier_zip_sha256"]
print("Dossier status:", lock["status"])
print("Freeze status:", freeze["status"])
print("ZIP:", zip_path)
print("ZIP SHA-256:", lock["dossier_zip_sha256"])
print("Next phase:", lock.get("next_phase"))
print("\nPHASE 0 REPRODUCIBILITY DOSSIER CERTIFIED AND FROZEN")


Dossier status: REPRODUCIBILITY_DOSSIER_CERTIFIED
Freeze status: FROZEN
ZIP: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/07_reproducibility/phase0/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_phase0_reproducibility_final_v100.zip
ZIP SHA-256: edf13a247759201197ce221690fcbf69dba2230900aeab0ecc1c54a064034108
Next phase: FASE_1_EVIDENCE_FOUNDATION

PHASE 0 REPRODUCIBILITY DOSSIER CERTIFIED AND FROZEN


In [11]:
# Autoverificação do ZIP extraído em diretório temporário
import tempfile
with tempfile.TemporaryDirectory() as td:
    td=Path(td)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(td)
    verify = td / "verify_dossier.py"
    result = subprocess.run([sys.executable, str(verify), "--dossier-root", str(td)], text=True, capture_output=True)
    print(result.stdout)
    assert result.returncode == 0, result.stderr


checked=765 failures=0

